# 노트북 4 — 6조건 학습 비교 (CSNN)

**EEG Resonate Encoding SNN 프로젝트** · 노트북 4

E1, E2, E3-정렬, E4-정렬, E4-균등, E4-어긋남 6조건 × 노트북 2를 통과한 피험자 × 시드 2개를 학습시키고
검증 정확도와 추론 1회당 총 스파이크 수를 CSV로 기록한다. 네트워크 구조는 CLAUDE.md 5절 명세에서
**절대 바꾸지 않는다** — 조건마다 바뀔 수 있는 것은 `forward_pass`에 들어가는 `spk_in` 하나뿐이다.

> 피험자는 `selected_subjects.json`(노트북 2)에서 정한 목록만 쓴다. 회당 2–3분 기준으로도 수 시간이 걸릴 수 있다.
> **코랩 GPU 런타임에서 실행할 것.** 이 노트북은 코드 작성만 하고, 실제 전체 실행은 코랩에서 진행한다.
> 결과는 (조건,피험자,시드)마다 CSV에 바로 이어붙여 저장된다 — 중간에 끊겨도 이미 끝난 실행은 안전하고,
> 같은 셀을 다시 실행하면 끝난 조합은 건너뛰고 남은 것만 이어서 돈다.

### 학습 목표
- 8장 쿡북 학습 루프(1에폭 부분데이터)를 다중 에폭 + 피험자 내 검증 분할로 확장한다.
- 가중치 초기화·데이터 분할을 조건간 동일하게 고정하고 인코더만 바꿔가며 비교한다.
- 검증 정확도와 추론 1회당 총 스파이크 수(입력+은닉+출력 모두 합산)를 기록해 CSV로 저장한다.


## 1. 설치와 임포트

In [1]:
%pip install -q snntorch

import json
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate
from snntorch import functional as SF
from snntorch import utils
import time

torch.manual_seed(0)                 # 기본 시드 (조건별로 다시 고정됨)
device = torch.device('cpu')         # 코랩에서는 'cuda'로 바꿔 사용


Note: you may need to restart the kernel to use updated packages.


## 2. 파라미터 설정

모든 조건이 공유하는 학습 설정과, 조건별로 달라지는 인코더 파라미터(노트북 3과 동일)를 정의한다.

In [2]:
N_DIMS = 8                       # 인코더 출력 채널(주파수/구간) 수
FS = 128                         # 샘플링 주파수 [Hz]
DECAY = 0.9                      # 공진/필터 감쇠율 (E3, E4 공용)

GRF_RANGE = (-3.0, 3.0)          # E1: z-score 값 범위
GRF_SIGMA_SCALE = 1.2            # E1: 가우시안 폭 배율
E2_THRESHOLDS = torch.tensor([0.1, 0.3, 0.6, 1.0])   # E2: 변화량 임계값 4단계

FREQS_ALIGNED = torch.logspace(
    torch.log10(torch.tensor(8.0)), torch.log10(torch.tensor(30.0)), N_DIMS
)                                                      # 정렬: 8-30 Hz 로그 배치 (mu·beta에 맞춤)
FREQS_UNIFORM = torch.linspace(2.0, 45.0, N_DIMS)      # 균등: 2-45 Hz 균등 배치 (사전지식 없음)
FREQS_MISALIGNED = torch.linspace(32.0, 45.0, N_DIMS)  # 어긋남: 32-45 Hz (mu·beta 회피)

E3_THRESHOLD = 1.5                # E3: 필터뱅크 delta 임계값
E4_THRESHOLD = 0.8                # E4: Resonate 진폭 임계값

SEEDS = [0, 1]                    # 시드 2개
TRAIN_RATIO = 0.8                 # 피험자 내 학습/검증 분할 비율
NUM_EPOCHS = 15                   # 다중 에폭 (8장 쿡북의 1에폭에서 확장)
BATCH_SIZE = 9                    # 배치 크기
LEARNING_RATE = 2e-3              # 학습률

SELECTED_SUBJECTS_PATH = '../data/processed/selected_subjects.json'  # 노트북 2의 선별 결과
OUT_PATH = '../data/processed/train_compare_results.csv'


## 3. 인코더 4종 구현 (노트북 3과 동일)

E1(Rate+GRF), E2(다중임계 Delta), E3/E4가 공유하는 지수윈도 STFT 선형 recursion, 그 위에 올린
필터벅크+Delta(E3) 규칙과 Resonate(E4) 규칙을 그대로 가져온다.

In [3]:
def encode_rate_grf(signal, num_neurons=N_DIMS, sigma_scale=GRF_SIGMA_SCALE, value_range=GRF_RANGE):
    # signal: (B,C,T) — z-score 값
    centers = torch.linspace(value_range[0], value_range[1], num_neurons)              # (N,) 수용장 중심
    sigma = (value_range[1] - value_range[0]) / num_neurons * sigma_scale              # 가우시안 폭

    x = signal[:, :, None, :]                                                          # (B,C,1,T)
    response = torch.exp(-((x - centers[None, None, :, None]) ** 2) / (2 * sigma ** 2))  # (B,C,N,T)
    spk = torch.bernoulli(response)                                                    # 반응률로 베르누이 발화
    return spk.permute(3, 0, 1, 2)                                                      # (T,B,C,N)


def encode_delta_multi(signal, thresholds=E2_THRESHOLDS):
    # signal: (B,C,T)
    delta = signal[:, :, 1:] - signal[:, :, :-1]
    delta = torch.cat([torch.zeros_like(delta[:, :, :1]), delta], dim=2)               # 첫 스텝 변화량 0
    pos = delta[:, :, None, :]                                                          # (B,C,1,T)
    thr = thresholds[None, None, :, None]                                              # (1,1,4,1)

    spk_pos = (pos > thr).float()                                                      # 상승 임계 4단계
    spk_neg = (pos < -thr).float()                                                     # 하강 임계 4단계
    spk = torch.cat([spk_pos, spk_neg], dim=2)                                         # (B,C,8,T)
    return spk.permute(3, 0, 1, 2)                                                      # (T,B,C,N)


def resonate_linear_state(signal, freqs, fs, decay):
    # E3, E4가 공유하는 선형 부분 — 지수 윈도우 STFT recursion
    omega = 2 * torch.pi * freqs / fs                          # 스텝당 위상 증가량
    cos_w, sin_w = torch.cos(omega), torch.sin(omega)
    u = torch.zeros(signal.size(0), freqs.numel(), signal.size(1))   # 실수부
    v = torch.zeros_like(u)                                           # 허수부
    u_rec, v_rec = [], []

    for step in range(signal.size(2)):
        x = signal[:, None, :, step]                                  # (B,1,C)
        u_next = decay * (cos_w[:, None] * u - sin_w[:, None] * v) + x
        v_next = decay * (sin_w[:, None] * u + cos_w[:, None] * v)
        u, v = u_next, v_next
        u_rec.append(u)
        v_rec.append(v)

    return torch.stack(u_rec), torch.stack(v_rec)                     # 각 (T,B,F,C)


def encode_filterbank_delta(u_rec, threshold=E3_THRESHOLD):
    # E3: 필터뱅크 실수부(u)에 델타 변조 적용 (통제군)
    delta = u_rec[1:] - u_rec[:-1]
    delta = torch.cat([torch.zeros_like(delta[:1]), delta], dim=0)
    spk = (delta.abs() > threshold).float()                           # (T,B,F,C)
    return spk.permute(0, 1, 3, 2)                                     # (T,B,C,F)


def encode_resonate(u_rec, v_rec, threshold=E4_THRESHOLD):
    # E4: 허수부 상향 영교차 + 실수부 임계 초과 (제안)
    v_prev = torch.cat([torch.zeros_like(v_rec[:1]), v_rec[:-1]], dim=0)
    spk = (v_prev <= 0).float() * (v_rec > 0).float() * (u_rec > threshold).float()
    return spk.permute(0, 1, 3, 2)                                     # (T,B,C,F)


def build_all_condition_inputs(signal):
    # signal: (B,C,T) — 한 피험자의 전체 시행 (z-score)
    u_aligned, v_aligned = resonate_linear_state(signal, FREQS_ALIGNED, FS, DECAY)
    u_uniform, v_uniform = resonate_linear_state(signal, FREQS_UNIFORM, FS, DECAY)
    u_misaligned, v_misaligned = resonate_linear_state(signal, FREQS_MISALIGNED, FS, DECAY)

    return {
        'E1': encode_rate_grf(signal),
        'E2': encode_delta_multi(signal),
        'E3_aligned': encode_filterbank_delta(u_aligned),
        'E4_aligned': encode_resonate(u_aligned, v_aligned),
        'E4_uniform': encode_resonate(u_uniform, v_uniform),
        'E4_misaligned': encode_resonate(u_misaligned, v_misaligned),
    }                                                                   # 각 값: (T,B,C,N)


## 4. 고정 네트워크 (CLAUDE.md 5절, 절대 변경 금지)

8장 CSNN 구조 기반이며 커널만 3×3(입력 맵이 9×8로 작아서)이다. 입력 9×8 → MaxPool2d(2) 두 번이면 4×4로 줄어
`32*4*4`가 된다. 가중치 초기화 시드가 조건간 동일하도록 `build_net`은 호출 시점에 시드를 다시 고정한다.

In [4]:
BETA = 0.9
SPIKE_GRAD = surrogate.fast_sigmoid()


def build_net(seed):
    torch.manual_seed(seed)                       # 가중치 초기화 고정 (조건 간 동일)
    net = nn.Sequential(
        nn.Conv2d(1, 12, 3, padding=1),
        snn.Leaky(beta=BETA, spike_grad=SPIKE_GRAD, init_hidden=True),
        nn.MaxPool2d(2),
        nn.Conv2d(12, 32, 3, padding=1),
        snn.Leaky(beta=BETA, spike_grad=SPIKE_GRAD, init_hidden=True),
        nn.Flatten(),
        nn.Linear(32 * 4 * 4, 2),
        snn.Leaky(beta=BETA, spike_grad=SPIKE_GRAD, init_hidden=True, output=True),
    ).to(device)
    return net


## 5. 순전파 함수 — 스파이크와 총 스파이크 수 기록

매 스텝 `spk_in[step]`을 넣어 시간축에 실제 정보가 흐르게 한다(8장과 달리 매 스텝 다른 입력).
입력·은닉두층·출력 스파이크를 모두 합산해 추론 1회당 총 스파이크 수(에너지 프록시용)를 같이 기록한다.

In [5]:
def forward_pass(net, spk_in):
    # spk_in: (T,B,C,N)
    utils.reset(net)                                       # 모든 LIF 막전위 초기화
    spk_out_rec = []
    total_spk_rec = []

    for step in range(spk_in.size(0)):
        x = spk_in[step].unsqueeze(1)                        # (B,1,C,N) 입력 스파이크
        cur1 = net[0](x)
        spk1 = net[1](cur1)                                   # 은닉1 스파이크
        pool1 = net[2](spk1)
        cur2 = net[3](pool1)
        spk2 = net[4](cur2)                                   # 은닉2 스파이크
        flat = net[5](spk2)
        cur3 = net[6](flat)
        spk_out, mem_out = net[7](cur3)                        # 출력 스파이크

        step_total = (
            x.sum(dim=(1, 2, 3)) + spk1.sum(dim=(1, 2, 3))
            + spk2.sum(dim=(1, 2, 3)) + spk_out.sum(dim=1)
        )                                                      # (B,) 이번 스텝 전체 스파이크 수
        spk_out_rec.append(spk_out)
        total_spk_rec.append(step_total)

    spk_out_rec = torch.stack(spk_out_rec)                     # (T,B,2)
    total_spikes = torch.stack(total_spk_rec).sum(dim=0)       # (B,) 시행당 총 스파이크 수
    return spk_out_rec, total_spikes


## 6. 학습·평가 함수 — 다중 에폭 + 검증 분할

8장 쿡북은 1에폭 부분데이터만 쓨지만, 여기서는 `NUM_EPOCHS`만큼 반복하고 검증셋으로 일반화 성능을 측정한다.
학습 종료 후 학습셋 정확도도 함께 재평가해 과적합 여부를 나중에 확인할 수 있게 하고, 검증 시행별 실제·예측 라벨도
따로 기록해 혼동행렬을 만들 수 있게 한다.

In [6]:
def train_and_eval(net, spk_train, y_train, spk_val, y_val):
    optimizer = torch.optim.Adam(net.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.999))
    loss_fn = SF.ce_rate_loss()
    n_train = spk_train.size(1)

    net.train()
    for epoch in range(NUM_EPOCHS):
        perm = torch.randperm(n_train)
        for start in range(0, n_train, BATCH_SIZE):
            idx = perm[start:start + BATCH_SIZE]
            spk_rec, _ = forward_pass(net, spk_train[:, idx])
            loss_val = loss_fn(spk_rec, y_train[idx])

            optimizer.zero_grad()
            loss_val.backward()
            optimizer.step()

    net.eval()
    with torch.no_grad():
        spk_rec_train, _ = forward_pass(net, spk_train)          # 학습셋 재평가 (과적합 확인용)
        train_acc = SF.accuracy_rate(spk_rec_train, y_train)

        spk_rec_val, total_spikes_val = forward_pass(net, spk_val)
        val_acc = SF.accuracy_rate(spk_rec_val, y_val)
        spikes_per_trial = total_spikes_val.float().mean().item()
        pred_val = spk_rec_val.sum(dim=0).argmax(dim=1)          # (검증 시행,) 예측 라벨

    y_true_list = y_val.cpu().tolist()
    y_pred_list = pred_val.cpu().tolist()
    return train_acc, val_acc, spikes_per_trial, y_true_list, y_pred_list


## 7. 선별 피험자 목록·데이터 불러오기와 피험자 내 분할

노트북 2가 저장한 `selected_subjects.json`에서 통과 피험자 목록과, 그 선별에 쓰인 창 길이(`num_steps`)·
npz 파일명을 그대로 가져온다. 학습/검증 분할은 torch의 가중치·인코딩 난수와 독립적인 numpy RNG
(피험자·시드로 고정)로 결정해, 조건이 바뀌어도 같은 (피험자,시드)에서는 동일한 분할이 나오도록 한다.

In [7]:
def make_split(n_trials, subject_id, seed):
    rng = np.random.default_rng(seed * 1000 + subject_id)   # 조건과 무관한 독립 RNG
    perm = rng.permutation(n_trials)
    n_train = int(n_trials * TRAIN_RATIO)
    return perm[:n_train], perm[n_train:]


with open(SELECTED_SUBJECTS_PATH, encoding='utf-8') as f:
    selection = json.load(f)

NUM_STEPS = selection['num_steps']                          # 노트북 2가 선택한 창 길이에 맞춤
SUBJECTS = selection['subjects']                            # 노트북 2를 통과한 피험자만
IN_PATH = '../data/processed/' + selection['npz_filename']

print(f"선택된 창 길이: {selection['window_label']} | 통과 피험자 수: {selection['n_selected']}")

npz = np.load(IN_PATH)
X = npz['X']                                                # (전체 시행, 9, NUM_STEPS)
y = npz['y']
subject_ids = npz['subject_ids']

print('X 형태:', X.shape)


선택된 창 길이: 0.5-3.5s | 통과 피험자 수: 10
X 형태: (1800, 9, 384)


## 8. 전체 실행 — 6조건 × 선별 피험자 × 시드 2개 (체크포인트 저장)

각 (조건,피험자,시드) 실행이 끝날 때마다 결과를 바로 CSV에 append한다. 시작할 때 이미 CSV에 있는 조합은
`done_keys`로 읽어 들여 건너뛰므로, 중간에 세션이 끊겨도 이 셀을 다시 실행하면 이어서 진행된다.

In [8]:
CONDITION_NAMES = ['E1', 'E2', 'E3_aligned', 'E4_aligned', 'E4_uniform', 'E4_misaligned']
RESULT_COLUMNS = [
    'condition', 'subject_id', 'seed', 'train_acc', 'val_acc',
    'spikes_per_trial', 'n_train', 'n_val', 'y_true', 'y_pred',
]
TOTAL_RUNS = len(SUBJECTS) * len(SEEDS) * len(CONDITION_NAMES)

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
pd.DataFrame(columns=RESULT_COLUMNS).to_csv(
    OUT_PATH, mode='a', header=not os.path.exists(OUT_PATH), index=False
)                                                              # 파일이 없으면 헤더만 만들고, 있으면 그대로 둠

done_df = pd.read_csv(OUT_PATH)
done_keys = set(zip(done_df['condition'], done_df['subject_id'], done_df['seed']))
print(f'이미 완료된 실행: {len(done_keys)}/{TOTAL_RUNS} (이어서 진행)')


def append_result_row(row):
    pd.DataFrame([row])[RESULT_COLUMNS].to_csv(OUT_PATH, mode='a', header=False, index=False)


start_time = time.time()
run_count = len(done_keys)

for subject_id in SUBJECTS:
    mask = subject_ids == subject_id
    signal = torch.tensor(X[mask], dtype=torch.float32)      # (45,9,NUM_STEPS)
    y_subj = y[mask]

    for seed in SEEDS:
        pending = [name for name in CONDITION_NAMES if (name, subject_id, seed) not in done_keys]

        train_idx, val_idx = make_split(len(y_subj), subject_id, seed)
        y_train = torch.tensor(y_subj[train_idx], dtype=torch.long)
        y_val = torch.tensor(y_subj[val_idx], dtype=torch.long)

        torch.manual_seed(seed)                               # 인코딩 난수(E1 베르누이 등) 고정
        condition_inputs = build_all_condition_inputs(signal)

        for condition_name in pending:
            spk_in = condition_inputs[condition_name]
            spk_train = spk_in[:, train_idx]
            spk_val = spk_in[:, val_idx]

            net = build_net(seed)                              # 조건마다 시드로 가중치 재고정
            train_acc, val_acc, spikes_per_trial, y_true_list, y_pred_list = train_and_eval(
                net, spk_train, y_train, spk_val, y_val
            )

            append_result_row({
                'condition': condition_name,
                'subject_id': subject_id,
                'seed': seed,
                'train_acc': train_acc,
                'val_acc': val_acc,
                'spikes_per_trial': spikes_per_trial,
                'n_train': len(train_idx),
                'n_val': len(val_idx),
                'y_true': ','.join(map(str, y_true_list)),        # 검증 시행별 실제 라벨
                'y_pred': ','.join(map(str, y_pred_list)),        # 검증 시행별 예측 라벨
            })

            run_count += 1
            elapsed = time.time() - start_time
            print(f'S{subject_id:02d} seed{seed} {condition_name:>13}: '
                  f'train_acc={train_acc*100:5.1f}% val_acc={val_acc*100:5.1f}% '
                  f'spikes={spikes_per_trial:8.0f} '
                  f'({run_count}/{TOTAL_RUNS}, 누적 {elapsed/60:.1f}분)')


이미 완료된 실행: 0/120 (이어서 진행)


S01 seed0            E1: train_acc= 52.8% val_acc= 44.4% spikes=   32104 (1/120, 누적 2.5분)


S01 seed0            E2: train_acc= 52.8% val_acc= 44.4% spikes=   24630 (2/120, 누적 5.2분)


S01 seed0    E3_aligned: train_acc= 52.8% val_acc= 44.4% spikes=   24182 (3/120, 누적 8.0분)


S01 seed0    E4_aligned: train_acc= 52.8% val_acc= 44.4% spikes=   17184 (4/120, 누적 10.7분)


...

S34 seed1            E2: train_acc= 50.0% val_acc= 33.3% spikes=   49202 (116/120, 누적 484.9분)


S34 seed1    E3_aligned: train_acc= 66.7% val_acc= 55.6% spikes=   56674 (117/120, 누적 487.0분)


S34 seed1    E4_aligned: train_acc= 50.0% val_acc= 33.3% spikes=   41018 (118/120, 누적 554.9분)


S34 seed1    E4_uniform: train_acc= 50.0% val_acc= 33.3% spikes=   40250 (119/120, 누적 557.1분)


S34 seed1 E4_misaligned: train_acc= 50.0% val_acc= 33.3% spikes=   39261 (120/120, 누적 559.2분)


## 9. 저장 결과 확인

결과는 8번 셀에서 이미 실행마다 CSV에 저장됐으므로, 여기서는 다시 읽어 행 수만 확인한다.

In [9]:
results_df = pd.read_csv(OUT_PATH)
print('저장 완료:', OUT_PATH, '| 행 수:', len(results_df), f'/ {TOTAL_RUNS}')
results_df.head()


저장 완료: ../data/processed/train_compare_results.csv | 행 수: 120 / 120


,condition,subject_id,seed,train_acc,val_acc,spikes_per_trial,n_train,n_val,y_true,y_pred
0,E1,1,0,0.527778,0.444444,32103.777344,36,9,"0,1,0,1,1,0,1,1,0","0,0,0,0,0,0,0,0,0"
1,E2,1,0,0.527778,0.444444,24629.777344,36,9,"0,1,0,1,1,0,1,1,0","0,0,0,0,0,0,0,0,0"
2,E3_aligned,1,0,0.527778,0.444444,24181.554688,36,9,"0,1,0,1,1,0,1,1,0","0,0,0,0,0,0,0,0,0"
3,E4_aligned,1,0,0.527778,0.444444,17183.554688,36,9,"0,1,0,1,1,0,1,1,0","0,0,0,0,0,0,0,0,0"
4,E4_uniform,1,0,0.527778,0.444444,17559.666016,36,9,"0,1,0,1,1,0,1,1,0","0,0,0,0,0,0,0,0,0"


## 정리

- E1, E2, E3-정렬, E4-정렬, E4-균등, E4-어긋남 6조건 × 피험자 10명 × 시드 2개 = 120회를 학습했다.
- 네트워크 구조·가중치 초기화 시드·데이터 분할은 조건간 동일하게 고정하고, 오직 인코더(`spk_in`)만 바꿔서 비교했다.
- 학습 정확도·검증 정확도와 추론 1회당 총 스파이크 수(입력+은닉+출력 모두 합산), 검증 시행별 실제·예측 라벨을
  `train_compare_results.csv`에 (조건,피험자,시드)마다 바로 append하며 저장했다(체크포인트).
- 중간에 끊겨도 같은 셀을 다시 실행하면 이미 끝난 조합은 건너뛰고 남은 것만 이어서 돈다.

다음 노트북(05_analysis)에서는 이 CSV로 조건별 정확도 평균·표준편차, 정확도–스파이크 수 트레이드오프, 대응표본 Wilcoxon 검정(E4 vs E3),
대역 정렬 3조건 비교를 수행한다.